In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr
from scipy.stats import chisquare
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [67]:
# Load data
df = pd.read_csv('dataset_v3.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (452, 35)

Columns: ['PHQ_class', 'age_class', 'A2_gender', 'D1_daily_screen', 'A3_religion', 'B1_residence', 'income_class', 'B4_family_type', 'B5_time_with_parents_hours', 'B6_relationship_with_family', 'weight_class', 'C3_physical_problems', 'D2_weekend_screen', 'D3_device', 'D4_content', 'D5_parents_screen_time', 'D6_ai_daily', 'D7_ai_impact', 'D8_creativity_decline', 'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime', 'E6_wake_time', 'F1_academic_satisfaction', 'F4_interest_reduced', 'G1_mood_swings', 'G2_anxious_without_device', 'G3_communication', 'G4_isolation', 'G5_negative_mental', 'G6_panic_attack', 'H1_mobile_while_eating', 'H2_appetite_change', 'BMI_class', 'D4_content_grouped']

First few rows:


,PHQ_class,age_class,A2_gender,D1_daily_screen,A3_religion,B1_residence,income_class,B4_family_type,B5_time_with_parents_hours,B6_relationship_with_family,...,G1_mood_swings,G2_anxious_without_device,G3_communication,G4_isolation,G5_negative_mental,G6_panic_attack,H1_mobile_while_eating,H2_appetite_change,BMI_class,D4_content_grouped
0,Moderate depressive,14-16 years,Female,3-5 hours,islam,Urban area,High Income (>300K),Nuclear family,2-6 hours,Good,...,Sometimes,Sometimes,Moderate,Very little time,Not sure,Sometimes,Always,Loss of appetite,Normal,Social Media & Mixed Usage
1,Moderate depressive,11-13 years,Female,3-5 hours,islam,Urban area,Low Income (≤150K),Nuclear family,2-6 hours,Good,...,Often,Sometimes,Bad,Very little time,Not sure,Sometimes,Always,Loss of appetite,Underweight,Other
2,Moderately Severe depressive,11-13 years,Female,1-3 hours,islam,Urban area,Low Income (≤150K),Nuclear family,2-6 hours,Good,...,Occasionally,Not at all,Moderate,Never,Never,Very little time,Often,Loss of appetite,Underweight,Entertainment & Media
3,Severe depressive,14-16 years,Female,Less than 1 hour,islam,Urban area,High Income (>300K),Nuclear family,Less than 2 hours,Moderate,...,Always,Occasionally,Bad,Never,Often,Often,Never,Overeating,Normal,Other
4,Moderately Severe depressive,14-16 years,Female,1-3 hours,islam,Urban area,High Income (>300K),Joint family,More than 6 hours,Bad,...,Always,Not at all,Good,Never,Often,Very little time,Often,Loss of appetite,Underweight,Other


# Screen_Time_vs_Mental_Health

In [68]:
from scipy.stats import chi2_contingency
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from datetime import datetime

# Fixed margins for PDF output
LEFT_MARGIN = 0.10
RIGHT_MARGIN = 0.90

def analysis(df, target, variables, section_title, save_image=True, filename="analysis_results.pdf"):
  
    results = []
    crosstabs = []
    
    for i, var in enumerate(variables, 1):
        ct = pd.crosstab(df[var], df[target], margins=True, margins_name='Total')
        crosstabs.append((var, ct))
        
        ct_test = pd.crosstab(df[var], df[target])
        chi2, p, dof, expected = chi2_contingency(ct_test)
        
        significance = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

        results.append({
            'Variable': var,
            'χ² Statistic': round(chi2, 3),
            'df': dof,
            'p-value': round(p, 4),
            'Sig.': significance
        })

    chi_square_table = pd.DataFrame(results)
    
    # Save all tables as multi-page A4 PDF with clean professional styling
    if save_image:
        plt.rcParams['font.family'] = 'Calibri'
        plt.rcParams['font.size'] = 10
        
        fig_width = 8.5
        fig_height = 11
        table_width = RIGHT_MARGIN - LEFT_MARGIN
        
        with PdfPages(filename) as pdf:
            page_num = 1
            
            # Page 1: Title/Header
            fig = plt.figure(figsize=(fig_width, fig_height))
            fig.patch.set_facecolor('white')
            ax = fig.add_subplot(111)
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            ax.text(0.5, 0.90, section_title, transform=ax.transAxes, 
                   fontsize=16, fontweight='bold', fontfamily='Calibri', 
                   va='top', ha='center', color='black')
            
            ax.text(LEFT_MARGIN, 0.80, "Outcome Variable:", transform=ax.transAxes,
                   fontsize=11, fontweight='bold', fontfamily='Calibri', va='top')
            ax.text(LEFT_MARGIN + 0.20, 0.80, target, transform=ax.transAxes,
                   fontsize=11, fontfamily='Calibri', va='top')
            
            ax.text(LEFT_MARGIN, 0.70, "Predictor Variables:", transform=ax.transAxes,
                   fontsize=11, fontweight='bold', fontfamily='Calibri', va='top')
            
            var_display = '\n'.join(variables)
            ax.text(LEFT_MARGIN, 0.68, var_display, transform=ax.transAxes,
                   fontsize=9, fontfamily='Calibri', va='top')
            
            ax.text(0.5, 0.03, f'Page {page_num}', transform=ax.transAxes, 
                   fontsize=9, fontfamily='Calibri', ha='center')
            page_num += 1
            
            pdf.savefig(fig, bbox_inches='tight', facecolor='white', dpi=300)
            plt.close()
            
            # Pages 2+: Individual tables
            for var, ct in crosstabs:
                fig = plt.figure(figsize=(fig_width, fig_height))
                fig.patch.set_facecolor('white')
                ax = fig.add_subplot(111)
                ax.axis('off')
                for spine in ax.spines.values():
                    spine.set_visible(False)
                
                ax.text(0.5, 0.96, f"Cross-tabulation: {var} × {target}", 
                       transform=ax.transAxes, fontsize=12, fontweight='bold', 
                       fontfamily='Calibri', ha='center', va='top')
                
                row_labels = list(ct.index)
                col_labels = list(ct.columns)
                cell_data = []
                
                for row_label in row_labels:
                    row_idx = list(ct.index).index(row_label)
                    row_data = [str(int(val)) for val in ct.values[row_idx]]
                    cell_data.append(row_data)
                
                full_data = []
                for i, label in enumerate(row_labels):
                    full_data.append([str(label)] + cell_data[i])
                
                all_col_labels = [var] + [str(col) for col in col_labels]
                
                table = ax.table(
                    cellText=full_data,
                    colLabels=all_col_labels,
                    cellLoc='center',
                    loc='upper center',
                    bbox=[0.15, 0.20, 0.70, 0.65]
                )
                table.auto_set_font_size(False)
                table.set_fontsize(8)
                
                for (row, col), cell in table.get_celld().items():
                    cell.set_text_props(fontfamily='Calibri', fontsize=9)
                    cell.set_facecolor('white')
                    cell.set_edgecolor('none')
                    cell.set_linewidth(0)
                    
                    if row == 0:
                        cell.set_facecolor('#f5f5f5')
                        cell.set_text_props(fontweight='bold', fontfamily='Calibri', fontsize=9, rotation=90)
                        cell.set_height(0.08)
                
                ax.text(0.92, 0.02, f'Page {page_num}', transform=ax.transAxes, 
                       fontsize=9, fontfamily='Calibri', ha='right')
                page_num += 1
                
                pdf.savefig(fig, bbox_inches='tight', facecolor='white', dpi=300)
                plt.close()
            
            # Final page: Chi-square summary
            fig = plt.figure(figsize=(fig_width, fig_height))
            fig.patch.set_facecolor('white')
            ax = fig.add_subplot(111)
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            ax.text(0.5, 0.96, f"Statistical Summary: Chi-Square Test Results", 
                   transform=ax.transAxes, fontsize=12, fontweight='bold', 
                   fontfamily='Calibri', ha='center', va='top')
            
            chi_df = chi_square_table
            table = ax.table(
                cellText=chi_df.values,
                colLabels=chi_df.columns,
                cellLoc='center',
                loc='upper center',
                bbox=[LEFT_MARGIN + 0.05, 0.20, 0.80, 0.70]
            )
            table.auto_set_font_size(False)
            table.set_fontsize(8)
            
            for (row, col), cell in table.get_celld().items():
                cell.set_text_props(fontfamily='Calibri', fontsize=9)
                cell.set_facecolor('white')
                cell.set_edgecolor('none')
                cell.set_linewidth(0)
                
                if row == 0:
                    cell.set_facecolor('#f5f5f5')
                    cell.set_text_props(fontweight='bold', fontfamily='Calibri', fontsize=9)
            
            ax.text(0.08, 0.10, "Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant",
                   transform=ax.transAxes, fontsize=8, fontfamily='Calibri', va='top')
            
            ax.text(0.92, 0.02, f'Page {page_num}', transform=ax.transAxes, 
                   fontsize=9, fontfamily='Calibri', ha='right')
            
            pdf.savefig(fig, bbox_inches='tight', facecolor='white', dpi=300)
            plt.close()
        
        plt.rcParams['font.family'] = plt.rcParamsDefault['font.family']
        print(f"✓ Report saved: {filename}")
    
    return chi_square_table

target = 'D1_daily_screen'
variables = ['PHQ_class','G1_mood_swings', 'G2_anxious_without_device',
          'B5_time_with_parents_hours', 'B6_relationship_with_family',
          'D3_device', 'G5_negative_mental', 'G6_panic_attack']


_ = analysis(df, target, variables, section_title="Screen_Time_vs_Mental_Health", filename="report_v4/1.Screen_Time_vs_Mental_Health.pdf")

✓ Report saved: report_v4/1.Screen_Time_vs_Mental_Health.pdf


# Screen Time VS Physical Health

In [69]:
target = 'D1_daily_screen'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Screen_Time_vs_Physical_Health", filename="report_v4/2.Screen_Time_vs_Physical_Health.pdf")

✓ Report saved: report_v4/2.Screen_Time_vs_Physical_Health.pdf
